# G12 GPU path: OLMoE W4A16 (llm-compressor GPTQ) served by vLLM, scored on eval_mini

Kaggle: Accelerator **GPU T4 x2**, Internet **on**. Colab: a GPU runtime (T4 works; L4 or A100 is faster).

1. Start with `LIMIT = 5` and `SMOKE = True`: run all cells interactively (about 1 h; most of it is GPTQ).
2. For the real run set `LIMIT = None`, `SMOKE = False` and use **Save Version → Save & Run All (Commit)**.
   The results zip is in that version's Output tab; unzip it into `teams/12-quantization/gpu_path/results/`.

Steps: download the bf16 checkpoint at the pinned revision → check it → quantize to W4A16 → check that the router
and `lm_head` stayed BF16 and every expert was quantized → score bf16 and W4A16 with the same vLLM settings →
paired comparison → `runs.csv` rows. Each step stops the run if it fails. Weights stay in the temp folder; only
their SHA-256 manifests go into the zip.

In [1]:
# ---- settings ----
LIMIT = None                 # items per task; None = the full eval_mini profile (1,870 items)
SMOKE = False              # True: 32 calibration samples (tests the pipeline); False: 256 x 1024 tokens
METHOD = "gptq"           # "gptq", or "rtn" (no calibration) if GPTQ fails on an expert
FORMATS = ["bf16", "W4A16"]
RUN_BY = "SomaKorada07"
REPO_URL = "https://github.com/SomaKorada07/era-v5-lab.git"
BRANCH = "g12-week2-soma"
MODEL = "allenai/OLMoE-1B-7B-0924"
REVISION = "6d84c48581ece794365f2b8e9cfb043c68ade9c5"
# Pinned to the versions the full run used (see results/environment.json).
QUANT_PKGS = ["llmcompressor==0.15.0", "compressed-tensors==0.20.1", "datasets==5.1.0"]
EVAL_PKGS = ["vllm==0.33.0", "lm_eval==0.4.13", "datasets==5.1.0"]

In [2]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working" if KAGGLE else "/content")
TMP = Path("/kaggle/tmp" if KAGGLE else "/content/tmp")      # big files, outside the 20 GB Kaggle output
OUT = WORK / "gpu_path_out"; RUNS = OUT / ("runs_limit%s" % LIMIT if LIMIT else "runs")
for d in (TMP, OUT, RUNS): d.mkdir(parents=True, exist_ok=True)

def sh(cmd, **kw):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, text=True, **kw)
    if p.returncode: raise SystemExit(f"failed ({p.returncode}): {cmd}")

REPO = TMP / "era-v5-lab"
if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
GP = REPO / "teams/12-quantization/gpu_path"
G9 = REPO / "teams/09-evaluation/harness/run_eval_mini.py"
gpus = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
                      shell=True, capture_output=True, text=True).stdout.strip()
print(gpus)
N_GPU = len(gpus.splitlines())
TP = 2 if N_GPU >= 2 else 1     # bf16 OLMoE (13.8 GB) does not fit one 16 GB T4 with a KV cache
env = {"started": time.strftime("%Y-%m-%dT%H:%M:%S"), "platform": "kaggle" if KAGGLE else "colab",
       "run_by": RUN_BY, "gpu": gpus, "tp": TP,
       "settings": dict(LIMIT=LIMIT, SMOKE=SMOKE, METHOD=METHOD, FORMATS=FORMATS, MODEL=MODEL, REVISION=REVISION,
                        BRANCH=BRANCH)}

$ git clone --depth 1 --branch g12-week2-soma https://github.com/SomaKorada07/era-v5-lab.git /kaggle/tmp/era-v5-lab


Cloning into '/kaggle/tmp/era-v5-lab'...


Tesla T4, 15360 MiB, 7.5
Tesla T4, 15360 MiB, 7.5


Updating files: 100% (773/773), done.


In [3]:
# two separate environments: llm-compressor and vLLM each pin their own torch and transformers
sh("pip -q install uv")
for name, pkgs in (("quant", QUANT_PKGS), ("eval", EVAL_PKGS)):
    venv = TMP / f"venv-{name}"
    if not (venv / "bin/python").exists():
        sh(f"uv venv -q --python 3.12 {venv}")
        sh(f"uv pip install -q --python {venv}/bin/python " + " ".join(f"'{p}'" for p in pkgs))
QPY, EPY = TMP / "venv-quant/bin/python", TMP / "venv-eval/bin/python"
versions = {}
for name, py, mods in (("quant", QPY, "llmcompressor compressed_tensors transformers torch datasets"),
                       ("eval", EPY, "vllm lm_eval transformers torch datasets")):
    code_ = "import importlib.metadata as m; print({k: m.version(k.replace('_','-')) for k in '%s'.split()})" % mods
    versions[name] = subprocess.run([str(py), "-c", code_], capture_output=True, text=True, check=True).stdout.strip()
    print(name, versions[name])
env["packages"] = versions

$ pip -q install uv
$ uv venv -q --python 3.12 /kaggle/tmp/venv-quant
$ uv pip install -q --python /kaggle/tmp/venv-quant/bin/python 'llmcompressor' 'datasets'
$ uv venv -q --python 3.12 /kaggle/tmp/venv-eval
$ uv pip install -q --python /kaggle/tmp/venv-eval/bin/python 'vllm' 'lm_eval==0.4.13' 'datasets==5.1.0'
quant {'llmcompressor': '0.15.0', 'compressed_tensors': '0.20.1', 'transformers': '5.21.0', 'torch': '2.14.1', 'datasets': '5.1.0'}
eval {'vllm': '0.33.0', 'lm_eval': '0.4.13', 'transformers': '5.20.2', 'torch': '2.14.0', 'datasets': '5.1.0'}


In [4]:
# bf16 checkpoint at the pinned revision
BF16 = TMP / "OLMoE-bf16"
if not (BF16 / "config.json").exists():
    sh(f"{QPY} -c \"from huggingface_hub import snapshot_download as s; "
       f"s('{MODEL}', revision='{REVISION}', local_dir='{BF16}', allow_patterns=['*.json','*.safetensors','*.txt'])\"")
sh(f"python {GP}/model_check.py {BF16} --label bf16 --out {OUT}/manifest_bf16.json")

$ /kaggle/tmp/venv-quant/bin/python -c "from huggingface_hub import snapshot_download as s; s('allenai/OLMoE-1B-7B-0924', revision='6d84c48581ece794365f2b8e9cfb043c68ade9c5', local_dir='/kaggle/tmp/OLMoE-bf16', allow_patterns=['*.json','*.safetensors','*.txt'])"


Fetching 9 files:  78%|███████▊  | 7/9 [00:47<00:16,  8.15s/it]

$ python /kaggle/tmp/era-v5-lab/teams/12-quantization/gpu_path/model_check.py /kaggle/tmp/OLMoE-bf16 --label bf16 --out /kaggle/working/gpu_path_out/manifest_bf16.json


Fetching 9 files: 100%|██████████| 9/9 [01:06<00:00,  7.35s/it]


{
 "router_dtypes": [
  "BF16"
 ],
 "router_tensors": 16,
 "output_dtype": "BF16",
 "experts_linear": 3072,
 "experts_packed": 0,
 "attn_linear": 64,
 "attn_packed": 0,
 "weight_bytes": 13838721960,
 "size_gb": 13.84,
 "base_params": 6919161856,
 "bpw": 16.0
}
wrote /kaggle/working/gpu_path_out/manifest_bf16.json
bf16: router ['BF16'], lm_head BF16: OK


In [5]:
W4 = TMP / (f"OLMoE-W4A16-{METHOD}" + ("-smoke" if SMOKE else ""))   # a smoke model is never reused for the real run
if not (W4 / "quantize_meta.json").exists():
    samples = "--samples 32 --seq-len 512" if SMOKE else "--samples 256 --seq-len 1024"
    sh(f"{QPY} {GP}/quantize_w4a16.py --model-dir {BF16} --out {W4} --method {METHOD} {samples}")
# fails unless the router and lm_head are BF16/F32 and every expert and attention projection is 4-bit
sh(f"python {GP}/model_check.py {W4} --label W4A16 --base {BF16} --expect-quantized --out {OUT}/manifest_W4A16.json")

$ /kaggle/tmp/venv-quant/bin/python /kaggle/tmp/era-v5-lab/teams/12-quantization/gpu_path/quantize_w4a16.py --model-dir /kaggle/tmp/OLMoE-bf16 --out /kaggle/tmp/OLMoE-W4A16-gptq --method gptq --samples 256 --seq-len 1024


Loading weights: 100%|██████████| 179/179 [00:29<00:00,  6.17it/s]


2026-11-03T13:35:12.7883 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-11-03T13:35:12.8930 | reset | INFO - Compression lifecycle reset
2026-11-03T13:35:13.0207 | apply_recipe_modifiers | WARNING - Detected an MoE model which has not been linearized. First load model `with llmcompressor.modeling.moe.linearize.load_quantizable_moe` before passing to `oneshot`. Falling back to post-load linearization.
2026-11-03T13:35:13.1459 | linearize_moe | WARNING - MoE is being linearized after loading in order to support efficient calibration of experts. However, this may be inefficient if the model checkpoint is already linearized (2D -> 3D -> 2D). Consider registering a load converter for faster load times. See https://docs.vllm.ai/projects/llm-compressor/en/latest/developer-tutorials/add-moe-support


Linearizing experts: 100%|██████████| 16/16 [00:06<00:00,  2.64it/s]


2026-11-03T13:35:20.0868 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 3136/3136 [00:00<00:00, 5286.92it/s]


... (37 outputs left out)

Compressing model: 100%|██████████| 3136/3136 [00:09<00:00, 341.11it/s]
/kaggle/tmp/venv-quant/lib/python3.12/site-packages/transformers/modeling_utils.py:3430: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:17<00:00, 17.33s/it]


2026-11-03T14:46:25.5309 | resave_config | INFO - Resaved original config with patched fields to /kaggle/tmp/OLMoE-W4A16-gptq/config.json


Dispatching model: 100%|██████████| 4327/4327 [00:00<00:00, 14678.41it/s]


{
 "model": "allenai/OLMoE-1B-7B-0924",
 "revision": "6d84c48581ece794365f2b8e9cfb043c68ade9c5",
 "method": "gptq",
 "scheme": "W4A16",
 "group_size": 128,
 "ignore": [
  "lm_head",
  "re:.*mlp.gate$"
 ],
 "calibration": {
  "dataset": "HuggingFaceFW/fineweb-edu/sample-10BT",
  "samples": 256,
  "seq_len": 1024,
  "prepare_s": 6.1
 },
 "quant_time_s": 4245.7,
 "gpu": "Tesla T4, 15360 MiB + Tesla T4, 15360 MiB",
 "versions": {
  "llmcompressor": "0.15.0",
  "compressed_tensors": "0.20.1",
  "transformers": "5.21.0",
  "torch": "2.14.1+cu130",
  "python": "3.12.3"
 },
 "finished": "2026-11-03T14:46:26+00:00"
}
$ python /kaggle/tmp/era-v5-lab/teams/12-quantization/gpu_path/model_check.py /kaggle/tmp/OLMoE-W4A16-gptq --label W4A16 --base /kaggle/tmp/OLMoE-bf16 --expect-quantized --out /kaggle/working/gpu_path_out/manifest_W4A16.json
{
 "router_dtypes": [
  "BF16"
 ],
 "router_tensors": 16,
 "output_dtype": "BF16",
 "experts_linear": 3072,
 "experts_packed": 3072,
 "attn_linear": 64,
 "attn

In [6]:
# same vLLM settings for both formats; each in its own process so the GPU memory is freed between them
DIRS = {"bf16": BF16, "W4A16": W4}
lim = f"--limit {LIMIT}" if LIMIT else ""
for fmt in FORMATS:
    t = time.time()
    sh(f"set -o pipefail; {EPY} {GP}/run_mini_vllm.py --model-dir {DIRS[fmt]} --format {fmt} --out {RUNS} --g9-harness {G9} "
       f"--tp {TP} {lim} 2>&1 | tee -a {OUT}/eval_{fmt}.log", executable="/bin/bash")
    env.setdefault("formats", {})[fmt] = {"wall_clock_s": round(time.time() - t)}

$ set -o pipefail; /kaggle/tmp/venv-eval/bin/python /kaggle/tmp/era-v5-lab/teams/12-quantization/gpu_path/run_mini_vllm.py --model-dir /kaggle/tmp/OLMoE-bf16 --format bf16 --out /kaggle/working/gpu_path_out/runs --g9-harness /kaggle/tmp/era-v5-lab/teams/09-evaluation/harness/run_eval_mini.py --tp 2  2>&1 | tee -a /kaggle/working/gpu_path_out/eval_bf16.log
W1103 14:47:09.698000 387 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1103 14:47:09.699000 387 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1103 14:47:09.700000 387 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=387) INFO 11-03 14:47:35 [startup.py:96] non-default args: {'dtype': 'float16', 'seed': 42, 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': '/kaggle/tmp/OLMoE-bf16'}
(MainProcess pid=

In [7]:
sh(f"python {REPO}/teams/12-quantization/eval_mini/paired.py {RUNS} --base bf16 --out {RUNS}/paired.md")
print(open(f"{RUNS}/paired.md").read())
if LIMIT is None:
    sh(f"python {GP}/add_runs_rows.py --runs {RUNS} --manifests {OUT}/manifest_bf16.json {OUT}/manifest_W4A16.json "
       f"> {OUT}/runs_rows.csv")
    print(open(f"{OUT}/runs_rows.csv").read())
shutil.copy(W4 / "quantize_meta.json", OUT / "quantize_meta.json")
env["finished"] = time.strftime("%Y-%m-%dT%H:%M:%S")
(OUT / "environment.json").write_text(json.dumps(env, indent=1))
zip_path = shutil.make_archive(str(WORK / "gpu_path_results"), "zip", OUT)
print("results:", zip_path)

$ python /kaggle/tmp/era-v5-lab/teams/12-quantization/eval_mini/paired.py /kaggle/working/gpu_path_out/runs --base bf16 --out /kaggle/working/gpu_path_out/runs/paired.md
Paired against **bf16** on the same items. Differences in points (format minus base); b = right in base, wrong in format; c = the reverse.

| Format | Task | n | Base | Format | Difference (95% CI) | b / c | McNemar p |
|---|---|---|---|---|---|---|---|
| W4A16 | arc_challenge | 500 | 46.8 | 47.2 | +0.4 (-1.9 to +2.7) | 16 / 18 | 0.864 |
| W4A16 | hellaswag | 400 | 66.0 | 66.0 | +0.0 (-2.0 to +2.0) | 8 / 8 | 1.000 |
| W4A16 | mmlu | 570 | 53.9 | 51.2 | -2.6 (-5.2 to -0.1) | 35 / 20 | 0.058 |
| W4A16 | gsm8k | 200 | 11.5 | 10.5 | -1.0 (-5.2 to +3.2) | 10 / 8 | 0.815 |
| W4A16 | truthfulqa_mc2 | 200 | 37.2 | 36.6 | -0.6 (-1.7 to +0.4) | - | - |
Paired against **bf16** on the same items. Differences in points (format minus base); b = right in base, wrong in format; c = the reverse.

| Format | Task | n | Base | Format | D